In [1]:
# ================================
# RAIN PREDICTION USING ML
# Logistic Regression – Exact Model
# ================================

# 1. IMPORT LIBRARIES
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Show plots inside notebook (for Jupyter)
# %matplotlib inline   # uncomment if using Jupyter


# 2. LOAD DATA
# --------------------------------------------------
# Ensure weatherAUS.csv is in the same folder
data = pd.read_csv('weatherAUS.csv')

print("First 5 rows:")
print(data.head())
print("\nLast 5 rows:")
print(data.tail())

print("\nData Info:")
print(data.info())

print("\nSummary statistics (numeric columns):")
print(data.describe())


# 3. MISSING VALUES ANALYSIS
# --------------------------------------------------
print("\nMissing value count per column:")
print(data.isnull().sum())

# Percentage of missing values, like Table 5.3
missing_percent = data.isnull().mean() * 100
missing_percent = missing_percent.round(1)
print("\nMissing value percentage (%):")
print(missing_percent.sort_values(ascending=False))

# Heatmap of missing data (Figure 5.1)
plt.figure(figsize=(10, 6))
sns.heatmap(data.isnull(), cbar=False, yticklabels=False)
plt.title("Missing Data Heatmap (Before Cleaning)")
plt.tight_layout()
plt.show()


# 4. DROP IRRELEVANT COLUMNS (as per project)
# --------------------------------------------------
cols_to_drop = ["Date", "Evaporation", "Sunshine", "Cloud9am", "Cloud3pm", "Location"]
for col in cols_to_drop:
    if col in data.columns:
        data.drop(columns=col, inplace=True)

print("\nColumns after dropping irrelevant ones:")
print(data.columns)


# 5. HANDLE MISSING VALUES
# --------------------------------------------------
# Project uses simple approach: drop rows with NaNs
data_clean = data.dropna()
print("\nShape before dropping NaNs:", data.shape)
print("Shape after  dropping NaNs:", data_clean.shape)

# Heatmap of missing data after cleaning (Figure 5.2)
plt.figure(figsize=(10, 6))
sns.heatmap(data_clean.isnull(), cbar=False, yticklabels=False)
plt.title("Missing Data Heatmap (After Cleaning)")
plt.tight_layout()
plt.show()


# 6. LABEL ENCODING FOR CATEGORICAL COLUMNS
# --------------------------------------------------
# According to report: WindGustDir, WindDir9am, WindDir3pm, RainToday, RainTomorrow
cat_cols = ["WindGustDir", "WindDir9am", "WindDir3pm", "RainToday", "RainTomorrow"]

le = LabelEncoder()
for col in cat_cols:
    if col in data_clean.columns:
        data_clean[col] = le.fit_transform(data_clean[col].astype(str))

print("\nData types after label encoding:")
print(data_clean.dtypes)


# 7. CORRELATION HEATMAP (Figure 5.3)
# --------------------------------------------------
plt.figure(figsize=(10, 8))
corr = data_clean.corr()
sns.heatmap(corr, annot=False, cmap="magma")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()


# 8. TEMP SCATTER PLOT (Figure 5.4)
# --------------------------------------------------
if {"MinTemp", "MaxTemp", "RainTomorrow"}.issubset(data_clean.columns):
    plt.figure(figsize=(8, 6))
    # color points by RainTomorrow
    sns.scatterplot(
        data=data_clean,
        x="MaxTemp",
        y="MinTemp",
        hue="RainTomorrow",
        palette="viridis",
        s=15
    )
    plt.title("MaxTemp vs MinTemp colored by RainTomorrow")
    plt.tight_layout()
    plt.show()
else:
    print("Columns for temperature scatter plot not found.")


# 9. TRAIN–TEST SPLIT
# --------------------------------------------------
# Separate features and target
X = data_clean.drop("RainTomorrow", axis=1)
y = data_clean["RainTomorrow"]

# As in your project: test_size = 0.20 (20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\nTrain shape:", X_train.shape, y_train.shape)
print("Test  shape:", X_test.shape, y_test.shape)


# 10. LOGISTIC REGRESSION MODEL
# --------------------------------------------------
log_reg = LogisticRegression(max_iter=1000)  # allow more iterations to ensure convergence
log_reg.fit(X_train, y_train)

# Predictions
y_pred = log_reg.predict(X_test)

# 11. EVALUATION
# --------------------------------------------------
# Accuracy
acc = accuracy_score(y_test, y_pred)
print("\nModel Accuracy:", round(acc, 4))

# Confusion Matrix & Heatmap
cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:\n", cm)

plt.figure(figsize=(4, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap="rocket", cbar=False,
            xticklabels=["No Rain", "Rain"],
            yticklabels=["No Rain", "Rain"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix – Logistic Regression")
plt.tight_layout()
plt.show()

# Classification report (Table 5.5 style)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=2))


FileNotFoundError: [Errno 2] No such file or directory: 'weatherAUS.csv'